# Finterion Charts — Python notebook demo

This notebook builds an OHLCV chart with an **RSI panel**, a **SuperTrend overlay**, and **buy/sell markers** from Python, then renders it inline.

Rendering is fully **self-contained** — `spec.display_in_jupyter()` returns an `<iframe srcdoc>` that bundles `@finterion/charts-react` from the wheel itself. No local server, no internet, no `EMBED_BASE` to configure.


## 1. Synthetic OHLCV data

A small random walk so the notebook has no external data dependency.

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 250

time = pd.date_range("2024-01-01", periods=n, freq="D")
drift = rng.normal(0, 1.2, size=n).cumsum()
open_ = 180.0 + drift
close = open_ + rng.normal(0, 1.0, size=n)
high = np.maximum(open_, close) + np.abs(rng.normal(0, 0.8, size=n))
low = np.minimum(open_, close) - np.abs(rng.normal(0, 0.8, size=n))
volume = 1_000_000 + rng.random(size=n) * 5_000_000

df = pd.DataFrame(
    {"time": time, "open": open_, "high": high, "low": low, "close": close, "volume": volume}
)
df.head()

,time,open,high,low,close,volume
0,2024-01-01,180.365660,181.456750,179.988251,179.997412,4.235359e+06
1,2024-01-02,179.117680,180.175383,178.878268,179.459235,2.711812e+06
2,2024-01-03,180.018221,182.322503,179.206167,181.746919,3.040783e+06
3,2024-01-04,181.146899,182.348901,178.521037,180.160042,3.200251e+06
4,2024-01-05,178.805656,181.177280,177.132244,178.560379,1.629035e+06


## 2. Compute RSI(14) and SuperTrend(ATR 10, ×3)

The binding does not ship indicator math — bring your own (`pandas-ta`, `ta-lib`, custom). These minimal implementations keep the notebook dependency-free.

In [3]:
def rsi(close: np.ndarray, period: int = 14) -> np.ndarray:
    out = np.full_like(close, np.nan, dtype=float)
    if close.size <= period:
        return out
    delta = np.diff(close)
    gain = np.where(delta > 0, delta, 0.0)
    loss = np.where(delta < 0, -delta, 0.0)
    avg_gain = gain[:period].mean()
    avg_loss = loss[:period].mean()
    out[period] = 100.0 - 100.0 / (1.0 + avg_gain / max(avg_loss, 1e-9))
    for i in range(period + 1, close.size):
        avg_gain = (avg_gain * (period - 1) + gain[i - 1]) / period
        avg_loss = (avg_loss * (period - 1) + loss[i - 1]) / period
        out[i] = 100.0 - 100.0 / (1.0 + avg_gain / max(avg_loss, 1e-9))
    return out


def supertrend(high: np.ndarray, low: np.ndarray, close: np.ndarray, atr_len: int = 10, factor: float = 3.0):
    n = close.size
    tr = np.zeros(n)
    tr[1:] = np.maximum.reduce([
        high[1:] - low[1:],
        np.abs(high[1:] - close[:-1]),
        np.abs(low[1:] - close[:-1]),
    ])
    atr = np.full(n, np.nan)
    if n > atr_len:
        atr[atr_len] = tr[1 : atr_len + 1].mean()
        for i in range(atr_len + 1, n):
            atr[i] = (atr[i - 1] * (atr_len - 1) + tr[i]) / atr_len

    line = np.full(n, np.nan)
    trend = np.zeros(n, dtype=int)
    signal = np.zeros(n, dtype=int)
    upper = np.full(n, np.nan)
    lower = np.full(n, np.nan)

    for i in range(n):
        if np.isnan(atr[i]):
            continue
        mid = (high[i] + low[i]) / 2.0
        upper[i] = mid + factor * atr[i]
        lower[i] = mid - factor * atr[i]
        if i > 0 and not np.isnan(upper[i - 1]):
            if close[i - 1] <= upper[i - 1]:
                upper[i] = min(upper[i], upper[i - 1])
            if close[i - 1] >= lower[i - 1]:
                lower[i] = max(lower[i], lower[i - 1])

        prev_trend = trend[i - 1] if i > 0 else 1
        if i == 0 or np.isnan(line[i - 1]):
            trend[i] = 1 if close[i] > mid else 0
        elif close[i] > upper[i - 1]:
            trend[i] = 1
        elif close[i] < lower[i - 1]:
            trend[i] = 0
        else:
            trend[i] = prev_trend

        line[i] = lower[i] if trend[i] == 1 else upper[i]
        if i > 0 and trend[i] != prev_trend and not np.isnan(line[i - 1]):
            signal[i] = 1 if trend[i] == 1 else -1
    return line, trend, signal


rsi14 = rsi(df["close"].to_numpy())
st_line, st_trend, st_signal = supertrend(
    df["high"].to_numpy(), df["low"].to_numpy(), df["close"].to_numpy()
)
st_up = np.where(st_trend == 1, st_line, np.nan)
st_dn = np.where(st_trend == 0, st_line, np.nan)

print(f"RSI: {np.sum(~np.isnan(rsi14))} valid points")
print(f"SuperTrend signals: {int((st_signal != 0).sum())} (buys={int((st_signal == 1).sum())}, sells={int((st_signal == -1).sum())})")

RSI: 236 valid points
SuperTrend signals: 6 (buys=3, sells=3)


## 3. Build the `ChartSpec`

Two panels — a candles `Price` panel with the SuperTrend trailing-stop drawn as two coloured overlays (bullish/bearish), and an `Indicator` panel for RSI with reference lines at 30/70. Buy/sell markers come from the SuperTrend regime flips.

In [4]:
from finterion_charts import (
    ChartSpec,
    Indicator,
    LineStyle,
    Marker,
    Price,
    SeriesType,
    Theme,
)

# Precompute ms-epoch once (pandas Timestamp.value is ns).
time_ms = (df["time"].astype("int64") // 1_000_000).to_numpy()

# Vectorise marker construction — iterate only over actual regime flips.
# Anchor the markers on the SuperTrend line itself (st_line) so they sit on
# the trailing-stop, not on the candle high/low.
buys = np.flatnonzero(st_signal == 1)
sells = np.flatnonzero(st_signal == -1)
markers = [
    Marker(time=int(time_ms[i]), side="buy", price=float(st_line[i]), label="B")
    for i in buys
] + [
    Marker(time=int(time_ms[i]), side="sell", price=float(st_line[i]), label="S")
    for i in sells
]

spec = (
    ChartSpec(
        theme=Theme.terminal_dark,
        # background="#131722",
        # grid_color="#494d57",
        grid="horizontal",
        initial_zoom=50,  # 100 = fully zoomed out, smaller = zoomed in
    )
    .with_bars(df=df)
    .with_columns({"rsi14": rsi14, "st_up": st_up, "st_dn": st_dn})
    .add_panel(
        Price(
            id="price",
            weight=3,
            title="AAPL · SuperTrend (ATR 10, ×3)",
            type=SeriesType.candles,
            overlays=[
                Indicator.line(
                    values="st_up",
                    color="#00ffa3",
                    glow="rgba(0,255,163,0.55)",
                    line_style=LineStyle.solid,
                ),
                Indicator.line(
                    values="st_dn",
                    color="#ff3d6e",
                    glow="rgba(255,61,110,0.55)",
                    line_style=LineStyle.solid,
                ),
            ],
        )
    )
    .add_panel(
        Indicator.panel(
            id="rsi",
            weight=1,
            title="RSI 14",
            values="rsi14",
            color="#a3ff12",
            ref_lines=[30, 70],
            y_range=(0, 100),
        )
    )
    .add_markers(markers)
    .validate()
)

print(f"panels: {[p.id for p in spec.panels]}")
print(f"markers: {len(spec.markers)} (buys={len(buys)}, sells={len(sells)})")

panels: ['price', 'rsi']
markers: 6 (buys=3, sells=3)


## 4. Render inline

`display_in_jupyter` returns an `<iframe srcdoc>` containing the bundled embed app and the spec injected as a JSON literal. Pass `base=...` to use a hosted embed iframe instead.


In [5]:
spec.display_in_jupyter(height=600)


In [6]:

from IPython.display import HTML, display


def hrow(*charts, gap_px: int = 12) -> None:
    """Display Finterion charts side-by-side in a horizontal flex row."""
    inner = "".join(c._repr_html_() for c in charts)
    display(HTML(
        f'<div style="display:flex;gap:{gap_px}px;align-items:stretch;'
        f'width:100%;flex-wrap:nowrap;">{inner}</div>'
    ))


# Same spec, two different zooms — illustrates the flex-row layout.
spec_zoomed = (
    ChartSpec(theme=Theme.terminal_dark, grid="horizontal", initial_zoom=25)
    .with_bars(df=df)
    .with_columns({"rsi14": rsi14, "st_up": st_up, "st_dn": st_dn})
    .add_panel(
        Price(
            id="price",
            weight=3,
            title="AAPL · last quarter (zoom=25)",
            type=SeriesType.candles,
            overlays=[
                Indicator.line(values="st_up", color="#00ffa3", glow="rgba(0,255,163,0.55)"),
                Indicator.line(values="st_dn", color="#ff3d6e", glow="rgba(255,61,110,0.55)"),
            ],
        )
    )
    .add_markers(markers)
    .validate()
)

hrow(
    spec.display_in_jupyter(width="25%", height=420),
    spec.display_in_jupyter(width="25%", height=420),
    spec.display_in_jupyter(width="25%", height=420),
    spec.display_in_jupyter(width="25%", height=420),
)

## 5. Other outputs

Same spec, different sinks:

In [6]:
# Persist to disk — a portable, validated artefact.
spec.to_json("aapl_supertrend.json", indent=2)

# Shareable embed URL (hash fragment, ~50KB for this dataset).
print(spec.embed_url(base=EMBED_BASE)[:140] + "…")

# Or open in your default browser (writes a temp HTML wrapper that postMessages the spec).
# spec.show(base=EMBED_BASE)

https://charts.finterion.com/embed/#spec=eyJ2ZXJzaW9uIjoxLCJkYXRhIjp7ImJhcnMiOnsidGltZSI6WzE3MDQwNjcyMDAuMCwxNzA0MTUzNjAwLjAsMTcwNDI0MDAwMC4…
